# Home Credit - Data Pipeline + df prep

Builds the dataset -> correct types, feature filtering, feature engineering, ...

**Source:** https://www.kaggle.com/code/greysky/home-credit-baseline

## 1. Imports & paths

In [1]:
import gc
from glob import glob
import numpy as np
import pandas as pd
import polars as pl

TRAIN_DIR = "data/train"
TEST_DIR = "data/test"
SAMPLE_DIR = "data/sample"

## 2. Define helper classes

### 2.1 Pipeline class
Handles data preprocessing steps:
- **set_table_dtypes**: Cast columns to appropriate data types (Int32, Float64, Date, String)
- **handle_dates**: Convert date columns to days relative to decision date
- **filter_cols**: Remove columns with >95% missing values or low variance (categorical with 1 or >200 unique values)

In [2]:
class Pipeline:
    @staticmethod
    def set_table_dtypes(df: pl.DataFrame):
        for col in df.columns:
            if col in ["case_id", "WEEK_NUM", "num_group1", "num_group2"]:
                df = df.with_columns(pl.col(col).cast(pl.Int32))
            elif col in ["date_decision"]:
                df = df.with_columns(pl.col(col).cast(pl.Date))
            elif col[-1] in ("P", "A"):
                df = df.with_columns(pl.col(col).cast(pl.Float64))
            elif col[-1] in ("M",):
                df = df.with_columns(pl.col(col).cast(pl.String))
            elif col[-1] in ("D",):
                df = df.with_columns(pl.col(col).cast(pl.Date))

        return df

    @staticmethod
    # Voor elke datumkolom, bereken het aantal dagen sinds de datum van beslissing
    def handle_dates(df: pl.DataFrame):
        for col in df.columns:
            if col[-1] in ("D",):
                df = df.with_columns(pl.col(col) - pl.col("date_decision"))
                df = df.with_columns(pl.col(col).dt.total_days())
                df = df.with_columns(pl.col(col).cast(pl.Float32))

        df = df.drop("date_decision", "MONTH")

        return df

    @staticmethod
    def filter_cols(df: pl.DataFrame):
        for col in df.columns:
            if col not in ["target", "case_id", "WEEK_NUM"]:
                isnull = df[col].is_null().mean()

                if isnull > 0.95:
                    df = df.drop(col)

        for col in df.columns:
            if (col not in ["target", "case_id", "WEEK_NUM"]) & (df[col].dtype == pl.String):
                freq = df[col].n_unique()

                if (freq == 1) | (freq > 200):
                    df = df.drop(col)

        return df

    @staticmethod
    def filter_correlated_cols(df: pd.DataFrame, threshold=0.95):
        """Remove one column from each pair of highly correlated columns (keeps non-numeric columns)"""
        # Protect important columns from being dropped
        protected_cols = ["target", "case_id", "WEEK_NUM"]

        # Get only numeric columns (excluding protected cols)
        numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns
                        if col not in protected_cols]

        # Calculate correlation matrix (only upper triangle for efficiency)
        corr_matrix = df[numeric_cols].corr(method='pearson').abs()
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        corr_upper = corr_matrix.where(mask)

        to_drop = [col for col in corr_upper.columns if any(
            corr_upper[col] > threshold)]

        # kwil leeftijd houden
        if "age" in to_drop:
            to_drop.remove("age")

        print(
            f"Dropping {len(to_drop)} highly correlated columns (threshold={threshold})")
        return df.drop(columns=to_drop)

### 2.2 Aggregator Class
Aggregates features by case_id at different depth levels. Creates maximum values for:
- Numeric features (columns ending in "P" or "A")
- Date features (columns ending in "D")
- String features (columns ending in "M")
- Other categorical features (columns ending in "T" or "L")
- Group count columns


In [3]:
class Aggregator:
    @staticmethod
    def num_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("P", "A")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def date_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("D",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def str_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("M",)]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def other_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if col[-1] in ("T", "L")]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def count_expr(df: pl.DataFrame):
        cols = [col for col in df.columns if "num_group" in col]

        expr_max = [pl.max(col).alias(f"max_{col}") for col in cols]

        return expr_max

    @staticmethod
    def get_exprs(df: pl.DataFrame):
        exprs = Aggregator.num_expr(df) + \
            Aggregator.date_expr(df) + \
            Aggregator.str_expr(df) + \
            Aggregator.other_expr(df) + \
            Aggregator.count_expr(df)

        return exprs

## 3. Data loading functions

- **read_file**: Reads a single parquet file and applies type casting and aggregation
- **read_files**: Reads multiple parquet files matching a glob pattern, concatenates them, and removes duplicates


In [4]:
def read_file(path, depth=None):
    df = pl.read_parquet(path)
    df = df.pipe(Pipeline.set_table_dtypes)

    if depth in [1, 2]:
        df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

    return df


def read_files(regex_path, depth=None):
    chunks = []
    for path in glob(str(regex_path)):
        df = pl.read_parquet(path)
        df = df.pipe(Pipeline.set_table_dtypes)

        if depth in [1, 2]:
            df = df.group_by("case_id").agg(Aggregator.get_exprs(df))

        chunks.append(df)

    df = pl.concat(chunks, how="vertical_relaxed")
    df = df.unique(subset=["case_id"])

    return df

## 4. Feature engineering

Combines base features with features from different data depths:
- Extracts month and weekday from decision date
- Joins multiple feature tables on case_id
- Converts date columns to relative days and handles missing values


In [5]:
def feature_eng(df_base: pl.DataFrame, depth_0, depth_1, depth_2) -> pl.DataFrame:
    df_base = (
        df_base
        .with_columns(
            month_decision=pl.col("date_decision").dt.month(),
            weekday_decision=pl.col("date_decision").dt.weekday(),
            year_decision=pl.col("date_decision").dt.year(),
        )
    )

    for i, df in enumerate(depth_0 + depth_1 + depth_2):
        df_base = df_base.join(df, how="left", on="case_id", suffix=f"_{i}")

    df_base = df_base.pipe(Pipeline.handle_dates)

    return df_base

## 5. Data format conversion

Converts polars DataFrame to pandas and converts object columns to categorical type for memory efficiency.


In [6]:
def to_pandas(df_data: pl.DataFrame, cat_cols=None) -> pd.DataFrame:
    df_data: pd.DataFrame = df_data.to_pandas()

    if cat_cols is None:
        cat_cols = list(df_data.select_dtypes("object").columns)

    df_data[cat_cols] = df_data[cat_cols].astype("category")

    return df_data, cat_cols

## 6. Load + prepare train en test data

Loads all training and testing data files, performs feature engineering, and prepares the dataset. Training data includes base information plus features from different depths (static, applications, tax registry, credit bureau, etc.).


In [7]:
# data_store = {
#     "df_base": read_file(f"{SAMPLE_DIR}/train_base_sampled.parquet"),
#     "depth_0": [
#         read_file(f"{SAMPLE_DIR}/train_static_cb_0_sampled.parquet"),
#         read_files(f"{SAMPLE_DIR}/train_static_0_*.parquet"),
#     ],
#     "depth_1": [
#         read_files(f"{SAMPLE_DIR}/train_applprev_1_*.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_tax_registry_a_1_sampled.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_tax_registry_b_1_sampled.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_tax_registry_c_1_sampled.parquet", 1),
#         read_files(f"{SAMPLE_DIR}/train_credit_bureau_a_1_*.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_credit_bureau_b_1_sampled.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_other_1_sampled.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_person_1_sampled.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_deposit_1_sampled.parquet", 1),
#         read_file(f"{SAMPLE_DIR}/train_debitcard_1_sampled.parquet", 1),
#     ],
#     "depth_2": [
#         read_file(f"{SAMPLE_DIR}/train_credit_bureau_b_2_sampled.parquet", 2),
#         read_files(f"{SAMPLE_DIR}/train_credit_bureau_a_2_*.parquet", 2),
#     ]
# }

# df_train = feature_eng(**data_store)
# print("train data shape:\t", df_train.shape)

data_store = {
    "df_base": read_file(f"{TRAIN_DIR}/train_base.parquet"),
    "depth_0": [
        read_file(f"{TRAIN_DIR}/train_static_cb_0.parquet"),
        read_files(f"{TRAIN_DIR}/train_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(f"{TRAIN_DIR}/train_applprev_1_*.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_tax_registry_a_1.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_tax_registry_b_1.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_tax_registry_c_1.parquet", 1),
        read_files(f"{TRAIN_DIR}/train_credit_bureau_a_1_*.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_credit_bureau_b_1.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_other_1.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_person_1.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_deposit_1.parquet", 1),
        read_file(f"{TRAIN_DIR}/train_debitcard_1.parquet", 1),
    ],
    "depth_2": [
        read_file(f"{TRAIN_DIR}/train_credit_bureau_b_2.parquet", 2),
        read_files(f"{TRAIN_DIR}/train_credit_bureau_a_2_*.parquet", 2),
    ]
}

df_train = feature_eng(**data_store)
print("train data shape:\t", df_train.shape)

train data shape:	 (1526659, 473)


In [8]:
data_store = {
    "df_base": read_file(f"{TEST_DIR}/test_base.parquet"),
    "depth_0": [
        read_file(f"{TEST_DIR}/test_static_cb_0.parquet"),
        read_files(f"{TEST_DIR}/test_static_0_*.parquet"),
    ],
    "depth_1": [
        read_files(f"{TEST_DIR}/test_applprev_1_*.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_a_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_b_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_tax_registry_c_1.parquet", 1),
        read_files(f"{TEST_DIR}/test_credit_bureau_a_1_*.parquet", 1),
        read_file(f"{TEST_DIR}/test_credit_bureau_b_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_other_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_person_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_deposit_1.parquet", 1),
        read_file(f"{TEST_DIR}/test_debitcard_1.parquet", 1),
    ],
    "depth_2": [
        read_file(f"{TEST_DIR}/test_credit_bureau_b_2.parquet", 2),
        read_files(f"{TEST_DIR}/test_credit_bureau_a_2_*.parquet", 2),
    ]
}

df_test = feature_eng(**data_store)
print("test data shape:\t", df_test.shape)

test data shape:	 (10, 472)


## 7. Filter features

Removes low-information columns from training data and keeps only the same features in test data.

Set an age rang of 18-100 bcs there were outliers/ages that are not possible

In [9]:
df_train = df_train.pipe(Pipeline.filter_cols)
df_test = df_test.select([col for col in df_train.columns if col != "target"])
print("train data shape:\t", df_train.shape)
print("test data shape:\t", df_test.shape)

train data shape:	 (1526659, 362)
test data shape:	 (10, 361)


In [10]:
df_train = df_train.with_columns(
    age=(abs(pl.col("dateofbirth_337D"))/365).round().cast(pl.UInt8))
df_test = df_test.with_columns(
    age=(abs(pl.col("dateofbirth_337D"))/365).round().cast(pl.UInt8))

df_train = df_train.filter((pl.col("age") >= 18) & (pl.col("age") <= 100))
df_test = df_test.filter((pl.col("age") >= 18) & (pl.col("age") <= 100))

print("train data shape:\t", df_train.shape)
print("test data shape:\t", df_test.shape)

train data shape:	 (1385624, 363)
test data shape:	 (10, 362)


## 8. Convert to pandas

Converts both datasets to pandas DataFrames and converts categorical columns to category dtype for memory efficiency.


In [11]:
df_train, cat_cols = to_pandas(df_train)
df_test, cat_cols = to_pandas(df_test, cat_cols)

# # Remove highly correlated columns (keeps all non-numeric columns) - TODO: misschien toch niet doen, want sommige modellen kunnen hier wel mee omgaan en het kan ook nuttige info bevatten (bv. max_payment_0_P en max_payment_1_P kunnen sterk gecorreleerd zijn maar toch allebei nuttig)
# df_train = Pipeline.filter_correlated_cols(df_train, threshold=0.95)
# df_test = df_test[[col for col in df_train.columns if col != "target"]]

# print("train data shape after correlation filter:\t", df_train.shape)
# print("test data shape after correlation filter:\t", df_test.shape)

del data_store
gc.collect()

C:\Users\Vik\AppData\Local\Temp\ipykernel_21092\464621053.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = list(df_data.select_dtypes("object").columns)


42

## 9. Save prepared datasets

In [12]:
df_train.to_parquet("data/df_train_full.parquet")
df_test.to_parquet("data/df_test_full.parquet")
print("Saved df_train and df_test to data/")

Saved df_train and df_test to data/
